In [1]:
print("Kernel is working!")

Kernel is working!


In [2]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/orders_train.txt"

df = pd.read_csv(DATA_PATH, sep=";")

print("Shape:", df.shape)
df.head()

Shape: (2325165, 15)


,orderID,orderDate,articleID,colorCode,sizeCode,productGroup,quantity,price,rrp,voucherID,voucherAmount,customerID,deviceID,paymentMethod,returnQuantity
0,a1000001,2014-01-01,i1000382,1972,44,3.0,1,10.00,29.99,0,0.0,c1010575,2,BPRG,0
1,a1000001,2014-01-01,i1000550,3854,44,3.0,1,20.00,39.99,0,0.0,c1010575,2,BPRG,0
2,a1000002,2014-01-01,i1001991,2974,38,8.0,1,35.00,49.99,0,0.0,c1045905,4,BPRG,0
3,a1000002,2014-01-01,i1001999,1992,38,8.0,1,49.99,49.99,0,0.0,c1045905,4,BPRG,1
4,a1000003,2014-01-01,i1001942,1968,42,8.0,1,10.00,35.99,0,0.0,c1089295,2,PAYPALVC,0


In [3]:
# Create binary return target
df["returned"] = (df["returnQuantity"] > 0).astype("int8")

# Convert order date
df["orderDate"] = pd.to_datetime(df["orderDate"])

# Date-based features
df["order_month"] = df["orderDate"].dt.month.astype("int8")
df["order_day"] = df["orderDate"].dt.day.astype("int8")
df["order_dayofweek"] = df["orderDate"].dt.dayofweek.astype("int8")
df["order_weekend"] = (df["order_dayofweek"] >= 5).astype("int8")

df[
    [
        "orderDate",
        "returned",
        "order_month",
        "order_day",
        "order_dayofweek",
        "order_weekend"
    ]
].head()

,orderDate,returned,order_month,order_day,order_dayofweek,order_weekend
0,2014-01-01,0,1,1,2,0
1,2014-01-01,0,1,1,2,0
2,2014-01-01,0,1,1,2,0
3,2014-01-01,1,1,1,2,0
4,2014-01-01,0,1,1,2,0


In [4]:
# Flag missing RRP
df["rrp_missing"] = df["rrp"].isna().astype("int8")

# Flag products sold above RRP
df["price_above_rrp"] = (df["price"] > df["rrp"]).astype("int8")

# Calculate discount
df["discount_pct"] = ((df["rrp"] - df["price"]) / df["rrp"]) * 100

# Remove invalid negative discounts
df.loc[df["discount_pct"] < 0, "discount_pct"] = 0

# Handle infinity and missing values
df["discount_pct"] = (
    df["discount_pct"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype("float32")
)

df[
    ["price", "rrp", "discount_pct", "price_above_rrp", "rrp_missing"]
].head(10)

,price,rrp,discount_pct,price_above_rrp,rrp_missing
0,10.00,29.99,66.655548,0,0
1,20.00,39.99,49.987495,0,0
2,35.00,49.99,29.985996,0,0
3,49.99,49.99,0.000000,0,0
4,10.00,35.99,72.214500,0,0
5,10.00,35.99,72.214500,0,0
6,25.00,39.99,37.484371,0,0
7,15.00,39.99,62.490623,0,0
8,0.00,59.99,100.000000,0,0
9,89.99,89.99,0.000000,0,0


In [5]:
df = df.sort_values(
    ["orderDate", "orderID"]
).reset_index(drop=True)

print("Data sorted successfully")
print(df.shape)

Data sorted successfully
(2325165, 23)


In [6]:
# Number of previous purchases of this product
df["product_previous_orders"] = (
    df.groupby("articleID").cumcount()
)

print("Product previous orders created successfully")

Product previous orders created successfully


In [7]:
# Number of previous returns for this product
df["product_previous_returns"] = (
    df.groupby("articleID")["returned"]
      .transform(lambda x: x.shift().cumsum())
      .fillna(0)
)

print("Product previous returns created successfully")

Product previous returns created successfully


In [8]:
df["product_historical_return_rate"] = (
    df["product_previous_returns"]
    / df["product_previous_orders"].replace(0, np.nan)
).fillna(0)

print("Product historical return rate created successfully")

Product historical return rate created successfully


In [9]:
df[
    [
        "articleID",
        "returned",
        "product_previous_orders",
        "product_previous_returns",
        "product_historical_return_rate"
    ]
].head(20)

,articleID,returned,product_previous_orders,product_previous_returns,product_historical_return_rate
0,i1000382,0,0,0.0,0.0
1,i1000550,0,0,0.0,0.0
2,i1001991,0,0,0.0,0.0
3,i1001999,1,0,0.0,0.0
4,i1001942,0,0,0.0,0.0
5,i1001942,0,1,0.0,0.0
6,i1001974,0,0,0.0,0.0
7,i1001976,0,0,0.0,0.0
8,i1002392,0,0,0.0,0.0
9,i1002457,1,0,0.0,0.0


In [10]:
df["customer_previous_orders"] = (
    df.groupby("customerID").cumcount()
)

print("Customer previous orders created successfully")

Customer previous orders created successfully


In [11]:
df["customer_previous_returns"] = (
    df.groupby("customerID")["returned"]
      .transform(lambda x: x.shift().cumsum())
      .fillna(0)
)

print("Customer previous returns created successfully")

Customer previous returns created successfully


In [13]:
df["customer_historical_return_rate"] = (
    df["customer_previous_returns"]
    / df["customer_previous_orders"].replace(0, np.nan)
).fillna(0)

print("Customer historical return rate created successfully")

Customer historical return rate created successfully


In [14]:
df[
    [
        "customerID",
        "returned",
        "customer_previous_orders",
        "customer_previous_returns",
        "customer_historical_return_rate"
    ]
].head(20)

,customerID,returned,customer_previous_orders,customer_previous_returns,customer_historical_return_rate
0,c1010575,0,0,0.0,0.0
1,c1010575,0,1,0.0,0.0
2,c1045905,0,0,0.0,0.0
3,c1045905,1,1,0.0,0.0
4,c1089295,0,0,0.0,0.0
5,c1089295,0,1,0.0,0.0
6,c1089295,0,2,0.0,0.0
7,c1089295,0,3,0.0,0.0
8,c1089295,0,4,0.0,0.0
9,c1050116,1,0,0.0,0.0


In [15]:
df[
    [
        "orderID",
        "orderDate",
        "customerID",
        "articleID",
        "returned",
        "customer_previous_orders",
        "customer_previous_returns"
    ]
].head(20)

,orderID,orderDate,customerID,articleID,returned,customer_previous_orders,customer_previous_returns
0,a1000001,2014-01-01,c1010575,i1000382,0,0,0.0
1,a1000001,2014-01-01,c1010575,i1000550,0,1,0.0
2,a1000002,2014-01-01,c1045905,i1001991,0,0,0.0
3,a1000002,2014-01-01,c1045905,i1001999,1,1,0.0
4,a1000003,2014-01-01,c1089295,i1001942,0,0,0.0
5,a1000003,2014-01-01,c1089295,i1001942,0,1,0.0
6,a1000003,2014-01-01,c1089295,i1001974,0,2,0.0
7,a1000003,2014-01-01,c1089295,i1001976,0,3,0.0
8,a1000003,2014-01-01,c1089295,i1002392,0,4,0.0
9,a1000004,2014-01-01,c1050116,i1002457,1,0,0.0


In [16]:
df.drop(
    columns=[
        "customer_previous_orders",
        "customer_previous_returns",
        "customer_historical_return_rate"
    ],
    inplace=True
)

print("Old customer history features removed.")

Old customer history features removed.


In [17]:
customer_orders = (
    df.groupby(
        ["customerID", "orderID", "orderDate"],
        as_index=False
    )
    .agg(
        order_items=("articleID", "size"),
        order_returns=("returned", "sum")
    )
)

customer_orders = customer_orders.sort_values(
    ["customerID", "orderDate", "orderID"]
).reset_index(drop=True)

customer_orders.head()

,customerID,orderID,orderDate,order_items,order_returns
0,c1000001,a1001465,2014-01-03,1,0
1,c1000001,a1004977,2014-01-07,1,1
2,c1000001,a1012517,2014-01-15,1,0
3,c1000001,a1013536,2014-01-16,2,2
4,c1000001,a1021517,2014-01-24,2,1


In [18]:
# Number of genuinely previous orders
customer_orders["customer_previous_orders"] = (
    customer_orders.groupby("customerID").cumcount()
)

# Previous purchased items
customer_orders["customer_previous_items"] = (
    customer_orders.groupby("customerID")["order_items"]
    .transform(lambda x: x.shift().cumsum())
    .fillna(0)
)

# Previous returned items
customer_orders["customer_previous_returns"] = (
    customer_orders.groupby("customerID")["order_returns"]
    .transform(lambda x: x.shift().cumsum())
    .fillna(0)
)

print("Customer history calculated.")

Customer history calculated.


In [19]:
customer_orders["customer_historical_return_rate"] = (
    customer_orders["customer_previous_returns"]
    / customer_orders["customer_previous_items"].replace(0, np.nan)
).fillna(0)

In [20]:
history_cols = [
    "customerID",
    "orderID",
    "customer_previous_orders",
    "customer_previous_items",
    "customer_previous_returns",
    "customer_historical_return_rate"
]

df = df.merge(
    customer_orders[history_cols],
    on=["customerID", "orderID"],
    how="left"
)

print("Customer history merged.")
print("Shape:", df.shape)

Customer history merged.
Shape: (2325165, 30)


In [21]:
df[
    [
        "orderID",
        "customerID",
        "articleID",
        "returned",
        "customer_previous_orders",
        "customer_previous_items",
        "customer_previous_returns",
        "customer_historical_return_rate"
    ]
].head(20)

,orderID,customerID,articleID,returned,customer_previous_orders,customer_previous_items,customer_previous_returns,customer_historical_return_rate
0,a1000001,c1010575,i1000382,0,0,0.0,0.0,0.0
1,a1000001,c1010575,i1000550,0,0,0.0,0.0,0.0
2,a1000002,c1045905,i1001991,0,0,0.0,0.0,0.0
3,a1000002,c1045905,i1001999,1,0,0.0,0.0,0.0
4,a1000003,c1089295,i1001942,0,0,0.0,0.0,0.0
5,a1000003,c1089295,i1001942,0,0,0.0,0.0,0.0
6,a1000003,c1089295,i1001974,0,0,0.0,0.0,0.0
7,a1000003,c1089295,i1001976,0,0,0.0,0.0,0.0
8,a1000003,c1089295,i1002392,0,0,0.0,0.0,0.0
9,a1000004,c1050116,i1002457,1,0,0.0,0.0,0.0


In [23]:
[col for col in df.columns if "product_" in col]

['product_previous_orders',
 'product_previous_returns',
 'product_historical_return_rate']

In [24]:
# Remove old product history columns if they exist
old_product_cols = [
    "product_previous_orders",
    "product_previous_items",
    "product_previous_returns",
    "product_historical_return_rate"
]

df.drop(
    columns=[c for c in old_product_cols if c in df.columns],
    inplace=True
)

print("Old product history columns cleared.")

Old product history columns cleared.


In [25]:
product_orders = (
    df.groupby(
        ["articleID", "orderID", "orderDate"],
        as_index=False
    )
    .agg(
        product_order_items=("returned", "size"),
        product_order_returns=("returned", "sum")
    )
)

product_orders = product_orders.sort_values(
    ["articleID", "orderDate", "orderID"]
).reset_index(drop=True)

print("Product-order rows:", len(product_orders))

Product-order rows: 1921178


In [26]:
product_orders["product_previous_orders"] = (
    product_orders.groupby("articleID").cumcount()
)

product_orders["product_previous_items"] = (
    product_orders.groupby("articleID")["product_order_items"]
    .transform(lambda x: x.shift().cumsum())
    .fillna(0)
)

product_orders["product_previous_returns"] = (
    product_orders.groupby("articleID")["product_order_returns"]
    .transform(lambda x: x.shift().cumsum())
    .fillna(0)
)

product_orders["product_historical_return_rate"] = (
    product_orders["product_previous_returns"]
    / product_orders["product_previous_items"].replace(0, np.nan)
).fillna(0)

print("Product features created successfully.")

Product features created successfully.


In [27]:
product_history_cols = [
    "articleID",
    "orderID",
    "product_previous_orders",
    "product_previous_items",
    "product_previous_returns",
    "product_historical_return_rate"
]

df = df.merge(
    product_orders[product_history_cols],
    on=["articleID", "orderID"],
    how="left"
)

print("Shape:", df.shape)

Shape: (2325165, 31)


In [28]:
print(
    df[
        [
            "orderID",
            "articleID",
            "returned",
            "product_previous_orders",
            "product_previous_items",
            "product_previous_returns",
            "product_historical_return_rate"
        ]
    ].head(20).to_string(index=False)
)

 orderID articleID  returned  product_previous_orders  product_previous_items  product_previous_returns  product_historical_return_rate
a1000001  i1000382         0                        0                     0.0                       0.0                             0.0
a1000001  i1000550         0                        0                     0.0                       0.0                             0.0
a1000002  i1001991         0                        0                     0.0                       0.0                             0.0
a1000002  i1001999         1                        0                     0.0                       0.0                             0.0
a1000003  i1001942         0                        0                     0.0                       0.0                             0.0
a1000003  i1001942         0                        0                     0.0                       0.0                             0.0
a1000003  i1001974         0                    

In [29]:
OUTPUT_PATH = "../data/processed/engineered_returns.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Processed dataset saved successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Processed dataset saved successfully!
Rows: 2325165
Columns: 31


In [30]:
import os

file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)

print(f"Saved file size: {file_size_mb:.2f} MB")

Saved file size: 345.57 MB
